In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
)
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from datasets import load_dataset

In [2]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="QuantFactory/Qwen2.5-0.5B-GGUF",
    filename="Qwen2.5-0.5B.Q4_K_M.gguf",
)

print(model_path)

/home/zaid/.cache/huggingface/hub/models--QuantFactory--Qwen2.5-0.5B-GGUF/snapshots/808e875628eb5b8d1e30728425039f4067b6ae16/Qwen2.5-0.5B.Q4_K_M.gguf


In [3]:
from llama_cpp import Llama


llm = Llama(
    model_path=model_path,
    n_ctx=1024,
    n_threads=8,
    verbose=False
)

In [4]:
def ask(prompt):

    out = llm(
        prompt,
        max_tokens=4096,
        temperature=0.7,
        stop=["<|im_end|>"]
    )

    return out["choices"][0]["text"]

In [5]:
ask("""You are generating training data for supervised fine-tuning.

Return ONLY valid JSON.

Schema:

{
    "messages":[
        {
            "role":"user",
            "content":"..."
        },
        {
            "role":"assistant",

            EITHER

            "content":"..."

            OR

            "tool_calls":[
                {
                    "tool":"...",
                    "arguments":{...}
                }
            ]
        }
    ]
}

Rules:

- Choose ONLY the requested tool.

- Use realistic biology questions.

- Questions should be diverse.

- For classify_sequence,
  include nucleotide sequences.

- For search_pubmed,
  search queries only.

- For gene_lookup,
  include realistic gene names.

- For no_tool,
  answer normally without tool_calls.

Return JSON only. """)

' Only the following tools are available: "Biology" and "Gene Lookup".  Please use only the tools listed in the table below.  If you don\'t need the tools listed below, please simply return an empty object.  If you need a different type of tool, please simply return an empty object, as long as the tool is listed in the table below.  If you need to return a list, please return a JSON array of the tools.  If you need to return a list of tools, please return only one tool.  If you need multiple tools, please return one tool.\n\nTABLE:\n\n| Tools | Description |\n| :-- :-- | :--: |\n| Biology | This tool queries a database of biological information. |\n| Gene Lookup | This tool searches a database of genes, and provides a list of genes related to a specific gene or sequence. |\n| Sequence | This tool queries a database of sequences. |\n| Sequence Query | This tool queries a database of sequences. |\n| Sequencing | This tool queries a database of sequencing data. |\n| Pubmed | This tool sea

## fine-tuning

In [5]:
MODEL_ID = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(MODEL_ID)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [6]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

In [7]:
model_1 = get_peft_model(model, lora_config)

In [9]:
dataset = load_dataset(
    "json",
    data_files="../datasets/v0.0.1/train.jsonl"
)
d1=dataset['train'][1]

In [12]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

import json

def format_example(example):

    user = example["messages"][0]["content"]

    assistant = example["messages"][1]

    if "tool_calls" in assistant:

        assistant_text = json.dumps(
            assistant["tool_calls"],
            ensure_ascii=False
        )

    else:

        assistant_text = assistant["content"]

    return (
        f"User: {user}\nAssistant: ",
        f"{assistant_text}"
    )
format_example(d1)

MAX_LENGTH = 512

def tokenize(example):

    prompt, answer = format_example(example)

    # Don't add BOS/EOS separately to each piece
    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
    )["input_ids"]

    answer_ids = tokenizer(
        answer + tokenizer.eos_token,
        add_special_tokens=False,
    )["input_ids"]

    # Concatenate into one sequence
    input_ids = prompt_ids + answer_ids

    # Truncate if necessary
    input_ids = input_ids[:MAX_LENGTH]

    # Labels:
    # Ignore prompt, learn only assistant response
    labels = (
        [-100] * len(prompt_ids)
        + answer_ids
    )[:MAX_LENGTH]

    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

PAD_ID = tokenizer.pad_token_id

def collate_fn(batch):

    max_len = max(len(sample["input_ids"]) for sample in batch)

    input_ids = []
    attention_masks = []
    labels = []

    for sample in batch:

        pad_len = max_len - len(sample["input_ids"])

        input_ids.append(
            sample["input_ids"] + [PAD_ID] * pad_len
        )

        attention_masks.append(
            sample["attention_mask"] + [0] * pad_len
        )

        labels.append(
            sample["labels"] + [-100] * pad_len
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }
#tokenize(d1)

In [14]:
print(tokenizer.eos_token)

<|endoftext|>


In [15]:
sample = tokenize(d1)

print(tokenizer.decode(sample["input_ids"]))
print(sample["labels"])

User: Can you tell me more about the BRCA1 gene?
Assistant: [{"tool": "gene_lookup", "arguments": {"gene_name": "BRCA1", "sequence": null, "query": null}}]<|endoftext|>
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 58, 4913, 14172, 788, 330, 42371, 27464, 497, 330, 16370, 788, 5212, 42371, 1269, 788, 330, 13095, 5049, 16, 497, 330, 15512, 788, 845, 11, 330, 1631, 788, 845, 3417, 60, 151643]


In [16]:
tokenized_dataset = dataset.map(
    tokenize,
    remove_columns=dataset['train'].column_names,
)
tokenized_dataset

Map:   0%|          | 0/29 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 29
    })
})

In [17]:
loader = DataLoader(
    dataset=tokenized_dataset['train'],
    batch_size=3,
    collate_fn=collate_fn
)

In [18]:
optimizer = torch.optim.Adam(params=model.parameters(),lr=3e-4)
loss = torch.nn.CrossEntropyLoss(ignore_index=-100)

In [19]:
NUM_EPOCHS = 4

model_1.train()

for epoch in range(NUM_EPOCHS):

    total_loss = 0.0

    for batch in loader:
        print("===")
        input_ids = batch["input_ids"].to(model_1.device)
        attention_mask = batch["attention_mask"].to(model_1.device)
        labels = batch["labels"].to(model_1.device)

        outputs = model_1(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs.loss

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}: "
        f"{total_loss / len(loader):.4f}"
    )

===
===
===
===
===
===
===
===
===
===
Epoch 1: 1.3304
===
===
===
===
===
===
===
===
===
===
Epoch 2: 0.1504
===
===
===
===
===
===
===
===
===
===
Epoch 3: 0.0557
===
===
===
===
===
===
===
===
===
===
Epoch 4: 0.0299


### ORIGINAL MODEL -- overriden by finetuned model

In [20]:
model.eval()
prompt = "User: What is a protein?\nAssistant: "
inputs = tokenizer(
    prompt,
    return_tensors="pt"
)


generated = model.generate(
    **inputs,
    max_new_tokens=128,
)
response = tokenizer.decode(
    generated[0],
    skip_special_tokens=True
)

print(response)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


User: What is a protein?
Assistant: [{"tool": "classify_sequence", "arguments": {"gene_name": null, "sequence": "null"}}]


### Fine tuned model

In [21]:
prompt = "User: Can you find recent research articles on CRISPR genome editing?\nAssistant: "
def ask_model(prompt=prompt, model=model_1):
    model.eval()
    
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )
    
    
    generated = model_1.generate(
        **inputs,
        max_new_tokens=128,
    )
    response = tokenizer.decode(
        generated[0],
        skip_special_tokens=True
    )
    
    print(response)
    return response

ask_model(model=model)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


User: Can you find recent research articles on CRISPR genome editing?
Assistant: [{"tool": "search_pubmed", "arguments": {"gene_name": null, "sequence": null, "query": "CRISPR genome editing recent research"}}]


'User: Can you find recent research articles on CRISPR genome editing?\nAssistant: [{"tool": "search_pubmed", "arguments": {"gene_name": null, "sequence": null, "query": "CRISPR genome editing recent research"}}]'

In [22]:
model_1.save_pretrained("../_checkpoints/qwen2.5-0.5b/exp001")
tokenizer.save_pretrained("../_checkpoints/qwen2.5-0.5b/exp001")

('../_checkpoints/qwen2.5-0.5b/exp001/tokenizer_config.json',
 '../_checkpoints/qwen2.5-0.5b/exp001/chat_template.jinja',
 '../_checkpoints/qwen2.5-0.5b/exp001/tokenizer.json')

In [29]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

# 1. Load the pristine, untouched base model
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

# 2. Inject your saved fine-tuned adapters right back into it
model_1 = PeftModel.from_pretrained(base_model, "../_checkpoints/qwen2.5-0.5b/exp001")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [30]:
ask_model(model=base_model)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


User: Can you find recent research articles on CRISPR genome editing?
Assistant: [{"tool": "search_pubmed", "arguments": {"gene_name": null, "sequence": null, "query": "CRISPR genome editing recent research"}}]
